# Análisis exploratorio y deserción en ClientesCDPSalud

Este notebook organiza el flujo de análisis en pasos cortos: inspección de las tres bases, construcción de una base unificada por cliente, modelado de deserción y una revisión de variables latentes. La variable objetivo se define con comportamiento futuro: deserción = no realizar Abono/Compra/Cambio de plan en la ventana de los siguientes meses. Primero se debe ejecutar el archivo analisis_desercion_salud.py para obtener el archivo merged_client_dataset.csv que resume las transacciones por cliente.

## 1. Librerías y configuración

En esta parte se importan las funciones del script auxiliar y se preparan algunas opciones de visualización para trabajar con tablas anchas.

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd

import analisis_desercion_salud as a

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 80)

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / 'outputs_desercion_notebook'
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Inspección rápida de las tres bases

Aquí se revisan muestras de las tres tablas para ver nombres de columnas, valores frecuentes y porcentaje de faltantes. Esto ayuda a decidir cómo unirlas y qué variables derivadas conviene crear.

In [8]:
socio_sample = a.read_sample(a.SOCIO_FILE)
clientes_sample = a.read_sample(a.CUSTOMER_FILE)
trans_sample = a.read_sample(a.TRANS_FILE)

a.print_eda('Sociodemograficos (muestra)', socio_sample)
a.print_eda('Clientes / movimientos (muestra)', clientes_sample)
a.print_eda('Transacciones (muestra)', trans_sample)

Sociodemograficos (muestra)
shape: (50000, 11)
missing ratio (top 12):
rangolineacredito    0.00008
escolaridad          0.00000
estado               0.00000
estadocivil          0.00000
estatus              0.00000
genero               0.00000
numerohijos          0.00000
puntualidad          0.00000
rangoedad            0.00000
rangoingreso         0.00000
ID_ficticio          0.00000
categorical highlights:
- escolaridad
escolaridad
Secundaria        16402
Preparatoria      11512
Primaria          10730
Profesional        6142
Carrera Tecnia     3891
No Estudio          983
Sin Datos           340
- estado
estado
ESTADO DE MEXICO    7491
DISTRITO FEDERAL    7111
JALISCO             5788
NUEVO LEON          3531
GUANAJUATO          2399
VERACRUZ            2163
GUERRERO            2117
TAMAULIPAS          2028
- estadocivil
estadocivil
Casado         21070
Soltero        20912
Union Libre     5892
Viudo           1092
Divorciado      1029
Sin Datos          5
- estatus
estatus
Activo

## 3. Construcción de la base unificada

En este paso se agregan las tablas largas a nivel de ID_ficticio, se unen con la información sociodemográfica y se genera una etiqueta de deserción por ventana futura de movimientos. Para cada cliente se toma una fecha de referencia y se verifica si hay Abono/Compra/Cambio de plan en los siguientes meses.

In [9]:
CHURN_HORIZON_MONTHS = 3
merged = pd.read_csv(BASE_DIR / 'outputs_desercion' / 'merged_client_dataset.csv', low_memory=False)

print('Forma de la base final:', merged.shape)
display(merged.head())

print('Distribucion de desercion:')
print(merged['desercion'].value_counts(dropna=False).to_string())

cols_target_candidates = [
    'ID_ficticio',
    'fecha_referencia_modelo',
    'fecha_vencimiento_objetivo',
    'fecha_inicio_ventana_renovacion',
    'fecha_fin_ventana_renovacion',
    'movimientos_ventana_renovacion',
    'renovo_en_ventana',
    'fecha_limite_ventana',
    'movimientos_siguientes_ventana',
    'tiene_movimientos_siguientes',
    'desercion',
]
cols_target = [col for col in cols_target_candidates if col in merged.columns]
display(merged[cols_target].head())

key_cols = [
    col for col in [
        'movimientos_total',
        'meses_pagados_sum',
        'dias_desde_ultimo_mov',
        'transacciones_total',
        'share_credito',
        'dias_desde_ultima_compra',
    ]
    if col in merged.columns
]

if key_cols:
    print('Faltantes en variables clave:')
    print(merged[key_cols].isna().mean().sort_values(ascending=False).to_string())

Forma de la base final: (609487, 105)


,escolaridad,estado,estadocivil,estatus,genero,numerohijos,puntualidad,rangoedad,rangoingreso,rangolineacredito,ID_ficticio,fecha_vencimiento_objetivo,fecha_referencia_modelo,fecha_inicio_ventana_renovacion,fecha_fin_ventana_renovacion,movimientos_ventana_renovacion,renovo_en_ventana,desercion,definicion_target,horizonte_desercion_meses,mov_abono,mov_cambio_de_plan,mov_compra,meses_pagados_sum,movimientos_total,meses_pagados_mean,plan_basico,plan_intermedio,canal_cajas_abono,canal_cajas_muebles,canal_cajas_ropa,fecha_primera_mov,fecha_ultima_mov,fecha_corte_max,fecha_vencimiento_max,n_planes,n_canales,share_abono,share_compra,share_cambio_plan,antiguedad_dias,dias_desde_ultimo_mov,dias_hasta_vencimiento,precio_vta_sum,precio_vta_mean,transacciones_total,trx_contado,trx_credito,cat_actividades_al_aire_libre,cat_alimentos_y_bebidas,cat_almacenamiento,cat_audio_y_video,cat_automotriz,cat_bebes,cat_belleza_y_cuidado_personal,cat_bicicletas_y_movilidad_eléctrica,cat_blancos,cat_caballero_exterior,cat_caballeros_accesorios,cat_celulares,cat_centro_de_belleza,cat_colchones_y_blancos,cat_cuidado_personal,cat_cuidado_personal_y_del_hogar,cat_cómputo_tablets_y_videojuegos,cat_dama_joven,cat_dama_señora,cat_decoración,cat_deportes_y_actividades_al_aire_libre,cat_deportes_y_movilidad,cat_deportivo,cat_deportivo.1,cat_enseres_domésticos,cat_escolar,cat_impulso,cat_interior_y_calceteria,cat_joyería_relojería_y_accesorios,cat_juguetes,cat_lencería,cat_línea_blanca,cat_maletas,cat_maquinaria_y_equipo_para_negocios,cat_mascotas,cat_mejoramiento_del_hogar,cat_motos_y_accesorios,cat_muebles_de_hogar_y_recamara,cat_niñas,cat_niños,cat_oficina_y_papeleria,cat_salud_y_bienestar,cat_servicios_digitales,cat_soluciones_coppel,cat_sin_descripcion,cat_terraza_y_jardin,cat_ventilación_y_calefacción,cat_zapato_caballero,cat_zapato_dama,cat_zapato_infantil,cat_óptica,fecha_ultima_compra,fecha_corte_transaccion_max,share_credito,share_contado,dias_desde_ultima_compra,n_categorias_desc
0,Profesional,GUERRERO,Soltero,Activos Sin Vencido,Femenino,1.0,A,De 51 a 55 Años,"5,000 o Menos","Mas de 45,000",287523,2025-10-20,2025-09-20,2025-10-21,2025-12-04,0,0,1,no_renovacion_seguro,1.5,3.0,0.0,1.0,4.0,4.0,1.0,4.0,0.0,4.0,0.0,0.0,2025-09-20,2025-09-20,2025-09-30,2026-01-20,1,1,0.750000,0.250000,0.0,10,10,112,24742.036992,10670.843975,71.0,20.0,51.0,0.0,0.0,0.0,0.0,0.0,17.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,7.0,4.0,4.0,1.0,0.0,1.0,0.0,0.0,0.0,2.0,3.0,1.0,4.0,16.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,3.0,0.0,0.0,2025-07-14,2025-07-31,0.718310,0.281690,17.0,17.0
1,Primaria,BAJA CALIFORNIA NORTE,Soltero,Clientes Z,Masculino,0.0,Z,De 26 a 30 Años,"5,000 o Menos","De 10,001 a 15,000",608607,2025-10-02,2025-04-02,2025-10-03,2025-11-16,0,0,1,no_renovacion_seguro,1.5,5.0,0.0,1.0,6.0,6.0,1.0,6.0,0.0,6.0,0.0,0.0,2025-04-02,2025-04-02,2025-04-30,2025-10-02,1,1,0.833333,0.166667,0.0,28,28,155,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Primaria,DISTRITO FEDERAL,Casado,Activos Sin Vencido,Femenino,2.0,A,De 46 a 50 Años,"De 5,001 a 10,000","Mas de 45,000",89769,2025-03-05,2025-01-15,2025-03-06,2025-04-19,0,0,1,no_renovacion_seguro,1.5,7.0,0.0,1.0,8.0,8.0,1.0,8.0,0.0,8.0,0.0,0.0,2024-07-02,2025-01-15,2025-01-31,2025-03-05,1,1,0.875000,0.125000,0.0,213,16,33,9510.515808,5570.479353,19.0,5.0,14.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2024-12-29,2024-12-31,0.736842,0.263158,2.0,10.0
3,Profesional,VERACRUZ,Soltero,Activos Sin Vencido,Femenino,0.0,A,De 21 a 25 Años,"De 5,001 a 10,000","De 35,001 a 40,000",166492,2025-11-15,2025-03-12,2025-11-16,2025-12-30,0,0,1,no_renovacion_seguro,

Distribucion de desercion:
desercion
1    520773
0     88714


,ID_ficticio,fecha_referencia_modelo,fecha_vencimiento_objetivo,fecha_inicio_ventana_renovacion,fecha_fin_ventana_renovacion,movimientos_ventana_renovacion,renovo_en_ventana,desercion
0,287523,2025-09-20,2025-10-20,2025-10-21,2025-12-04,0,0,1
1,608607,2025-04-02,2025-10-02,2025-10-03,2025-11-16,0,0,1
2,89769,2025-01-15,2025-03-05,2025-03-06,2025-04-19,0,0,1
3,166492,2025-03-12,2025-11-15,2025-11-16,2025-12-30,0,0,1
4,193790,2025-10-10,2025-11-10,2025-11-11,2025-12-25,0,0,1


Faltantes en variables clave:
transacciones_total         0.221097
share_credito               0.221097
dias_desde_ultima_compra    0.221097
movimientos_total           0.000000
meses_pagados_sum           0.000000
dias_desde_ultimo_mov       0.000000


## 4. Exploración visual de la deserción

Antes de modelar conviene revisar el desbalance de la variable objetivo. En este caso la figura se guarda en disco para no depender del backend del notebook.

In [10]:
a.plot_target_distribution(merged, OUTPUT_DIR)
print('Figura guardada en:', OUTPUT_DIR / 'target_distribution.png')

Figura guardada en: /Users/ivangm/Documents/CIMAT-MAESTRIA/4-Semestre/Consultoría/Coppel/outputs_desercion_notebook/target_distribution.png


## 5. Dos maneras de modelar la deserción

Se comparan dos enfoques: uno interpretable con regresión logística y otro no lineal con HistGradientBoosting. El primero sirve para explicar drivers; el segundo suele capturar mejor interacciones y umbrales.

In [11]:
logistic_metrics = a.fit_logistic_model(merged)
gbdt_metrics = a.fit_hist_gradient_boosting_model(merged)

metrics = pd.DataFrame([logistic_metrics, gbdt_metrics])
display(metrics)
metrics.to_csv(OUTPUT_DIR / 'metricas_modelos.csv', index=False)

LogisticRegression (baseline interpretable)
{'model': 'LogisticRegression (baseline interpretable)', 'roc_auc': 0.8056388675376807, 'avg_precision': 0.9578776699640152, 'f1': 0.822413878399103}
[[ 4295  1527]
 [ 9242 24936]]
              precision    recall  f1-score   support

           0     0.3173    0.7377    0.4437      5822
           1     0.9423    0.7296    0.8224     34178

    accuracy                         0.7308     40000
   macro avg     0.6298    0.7337    0.6331     40000
weighted avg     0.8513    0.7308    0.7673     40000

HistGradientBoosting (nonlinear alternative)
{'model': 'HistGradientBoosting (nonlinear alternative)', 'roc_auc': 0.8293093913994709, 'avg_precision': 0.9643376661014698, 'f1': 0.9262236414689203}
[[  994  4828]
 [  532 33646]]
              precision    recall  f1-score   support

           0     0.6514    0.1707    0.2705      5822
           1     0.8745    0.9844    0.9262     34178

    accuracy                         0.8660     40000
  

,model,roc_auc,avg_precision,f1
0,LogisticRegression (baseline interpretable),0.805639,0.957878,0.822414
1,HistGradientBoosting (nonlinear alternative),0.829309,0.964338,0.926224


## 6. Variables latentes

Aquí se prueba si hay una estructura latente útil en indicadores conductuales agregados. Sirve como primera evidencia de si existen dimensiones subyacentes de riesgo o permanencia.

In [13]:
latent_loadings = a.latent_variable_analysis(merged)
if not latent_loadings.empty:
    display(latent_loadings)
    latent_loadings.to_csv(OUTPUT_DIR / 'latent_loadings.csv')

    # Reporting-friendly interpretation table
    sq = latent_loadings.pow(2)
    communality = sq.sum(axis=1)
    uniqueness = (1 - communality).clip(lower=0)
    primary_factor = latent_loadings.abs().idxmax(axis=1)
    primary_loading = latent_loadings.apply(lambda row: row.loc[row.abs().idxmax()], axis=1)

    # Variable contribution percentage inside each factor
    factor_contrib_pct = sq.div(sq.sum(axis=0), axis=1).mul(100)
    factor_contrib_pct.columns = [f'{c}_contrib_pct' for c in factor_contrib_pct.columns]

    interpretation = pd.DataFrame({
        'primary_factor': primary_factor,
        'primary_loading': primary_loading,
        'communality': communality,
        'uniqueness': uniqueness,
    }, index=latent_loadings.index)

    interpretation = interpretation.join(factor_contrib_pct)
    interpretation = interpretation.sort_values(
        by=['primary_factor', 'primary_loading'],
        ascending=[True, False],
    )

    print('Interpretation table (primary factor, uniqueness, contribution %)')
    display(interpretation)
    interpretation.to_csv(OUTPUT_DIR / 'latent_interpretation_table.csv')

Factor analysis loadings
                          factor_1  factor_2
movimientos_total         0.999264 -0.002997
meses_pagados_sum         0.999048 -0.003448
dias_hasta_vencimiento    0.663563 -0.064114
antiguedad_dias           0.638319  0.065531
share_abono               0.623411 -0.011018
precio_vta_mean           0.139047  0.921230
n_categorias_desc         0.127463  0.967815
transacciones_total       0.110782  0.824587
n_canales                 0.064662  0.010833
n_planes                  0.049134  0.000227
share_contado             0.031566  0.117457
dias_desde_ultimo_mov     0.028066  0.012722
share_cambio_plan         0.008372 -0.000128
meses_pagados_mean       -0.004201 -0.009429
dias_desde_ultima_compra -0.013777 -0.014260
share_credito            -0.031566 -0.117457
share_compra             -0.623531  0.011020
AUC using only latent risk score: 0.7112


,factor_1,factor_2
movimientos_total,0.999264,-0.002997
meses_pagados_sum,0.999048,-0.003448
meses_pagados_mean,-0.004201,-0.009429
share_abono,0.623411,-0.011018
share_compra,-0.623531,0.011020
share_cambio_plan,0.008372,-0.000128
antiguedad_dias,0.638319,0.065531
dias_desde_ultimo_mov,0.028066,0.012722
dias_hasta_vencimiento,0.663563,-0.064114
transacciones_total,0.110782,0.824587


Interpretation table (primary factor, uniqueness, contribution %)


,primary_factor,primary_loading,communality,uniqueness,factor_1_contrib_pct,factor_2_contrib_pct
movimientos_total,factor_1,0.999264,0.998537,0.001463,27.138862,3.590023e-04
meses_pagados_sum,factor_1,0.999048,0.998108,0.001892,27.127112,4.750104e-04
dias_hasta_vencimiento,factor_1,0.663563,0.444427,0.555573,11.967300,1.642871e-01
antiguedad_dias,factor_1,0.638319,0.411745,0.588255,11.074053,1.716286e-01
share_abono,factor_1,0.623411,0.388762,0.611238,10.562811,4.851976e-03
n_canales,factor_1,0.064662,0.004299,0.995701,0.113641,4.690076e-03
n_planes,factor_1,0.049134,0.002414,0.997586,0.065615,2.060674e-06
dias_desde_ultimo_mov,factor_1,0.028066,0.000950,0.999050,0.021409,6.468036e-03
share_cambio_plan,factor_1,0.008372,0.000070,0.999930,0.001905,6.561131e-07
share_compra,factor_1,-0.623531,0.388912,0.611088,10.566878,4.853560e-03


## 7. Cierre

La salida principal del notebook es una base unificada por cliente, una etiqueta operativa de deserción basada en comportamiento futuro y una comparación entre dos modelos.